# Phishing Detection — Dataset Reproduction Experiment

This notebook continues the final project using the downloaded `dataset_phishing.csv` file.

The original GitHub project is used as the selected source for critical evaluation, but because its processed CSV file is missing, this notebook uses a public phishing dataset with similar purpose: detecting whether a URL is `legitimate` or `phishing`.

## 1. Imports and Settings

In [ ]:
import zipfile
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

## 2. Data Loading

Put the dataset in the `data/` folder.

Supported options:
- `data/dataset_phishing.csv`
- `data/dataset_phishing.csv.zip`

In [ ]:
DATA_DIR = Path("data")
CSV_PATH = DATA_DIR / "dataset_phishing.csv"
ZIP_PATH = DATA_DIR / "dataset_phishing.csv.zip"

if not CSV_PATH.exists():
    if ZIP_PATH.exists():
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            z.extractall(DATA_DIR)
    else:
        raise FileNotFoundError(
            "Dataset not found. Please place dataset_phishing.csv or "
            "dataset_phishing.csv.zip inside the data/ folder."
        )

df = pd.read_csv(CSV_PATH)
print("Dataset shape:", df.shape)
display(df.head())

## 3. Data Inspection

We inspect:
- dataset size
- columns and data types
- missing values
- duplicate rows
- target distribution
- constant features

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.value_counts())

print("\nMissing values:")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
display(df["status"].value_counts())
display((df["status"].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Convert target to numeric label:
# legitimate = 0, phishing = 1
df["label"] = df["status"].map({"legitimate": 0, "phishing": 1})

if df["label"].isna().any():
    raise ValueError("Some labels could not be mapped. Check values in the status column.")

print(df[["status", "label"]].head())

In [ ]:
feature_cols = [c for c in df.columns if c not in ["url", "status", "label"]]
X_all = df[feature_cols]
y = df["label"]

constant_cols = [c for c in feature_cols if df[c].nunique(dropna=False) <= 1]
print("Constant columns:", constant_cols)
print("Number of constant columns:", len(constant_cols))

## 4. Temporal Analysis

The assignment asks for temporal analysis when temporal features exist.  
This dataset does not appear to include explicit date/time columns, so temporal drift cannot be evaluated directly. This is a limitation because phishing tactics and benign URL patterns can change over time.

In [ ]:
possible_time_cols = [
    c for c in df.columns
    if any(token in c.lower() for token in ["date", "time", "timestamp", "created", "updated"])
]
print("Possible temporal columns:", possible_time_cols)

## 5. Exploratory Data Analysis

The dataset is balanced if the two classes have similar counts.  
Class imbalance is important in cybersecurity because a high accuracy may hide poor detection of attacks.

In [ ]:
class_counts = df["status"].value_counts()

plt.figure(figsize=(5, 4))
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

In [ ]:
# Basic statistics for numeric features
display(df[feature_cols].describe().T)

In [ ]:
important_features = [
    "length_url", "length_hostname", "ip", "nb_dots", "nb_hyphens",
    "nb_qm", "nb_eq", "ratio_digits_url", "phish_hints",
    "nb_hyperlinks", "web_traffic", "google_index", "page_rank"
]
important_features = [c for c in important_features if c in df.columns]

for col in important_features:
    plt.figure(figsize=(6, 4))
    plt.hist(df.loc[df["label"] == 0, col].dropna(), bins=40, alpha=0.6, label="legitimate")
    plt.hist(df.loc[df["label"] == 1, col].dropna(), bins=40, alpha=0.6, label="phishing")
    plt.title(f"Distribution of {col} by class")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.legend()
    plt.show()

## 6. Correlation Analysis

We use Spearman correlation because many URL and webpage features are counts, indicators, or skewed variables. Spearman correlation is rank-based and is more robust to outliers than Pearson correlation.

In [ ]:
corr = df[feature_cols + ["label"]].corr(method="spearman")["label"].drop("label")
top_corr = corr.reindex(corr.abs().sort_values(ascending=False).index)

display(top_corr.head(20))

plt.figure(figsize=(8, 6))
top_corr.head(15).sort_values().plot(kind="barh")
plt.title("Top Spearman Correlations with Phishing Label")
plt.xlabel("Spearman correlation")
plt.show()

## 7. Outlier Analysis

URL and cyber features often have heavy-tailed and skewed distributions.  
We use IQR as a robust method for estimating outlier rates.

In [ ]:
def outlier_rate_iqr(series: pd.Series) -> float:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return 0.0
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).mean()

outlier_rates = pd.Series({col: outlier_rate_iqr(df[col]) for col in feature_cols})
display((outlier_rates.sort_values(ascending=False) * 100).round(2).head(20))

## 8. Feature Engineering and Preprocessing

The dataset already contains engineered features extracted from URLs and webpage properties.

We still perform preprocessing:
- remove non-numeric fields from the model input (`url`, `status`)
- map `status` to numeric target `label`
- remove constant columns
- use feature scaling for Logistic Regression
- use tree-based models without scaling

In [ ]:
X = df[feature_cols].drop(columns=constant_cols, errors="ignore").copy()
X = X.fillna(X.median(numeric_only=True))

print("Final feature matrix shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain target distribution:")
display(y_train.value_counts(normalize=True).sort_index())
print("\nTest target distribution:")
display(y_test.value_counts(normalize=True).sort_index())

## 9. Model Training

The assignment requires at least two models.  
We train four models:
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Extra Trees

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=1000,
            solver="liblinear",
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=18,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=2,
        random_state=RANDOM_STATE
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100,
        max_depth=18,
        min_samples_leaf=2,
        class_weight="balanced",
        n_jobs=2,
        random_state=RANDOM_STATE
    )
}

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    result = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_score) if y_score is not None else np.nan,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print(classification_report(y_test, y_pred, target_names=["legitimate", "phishing"], digits=4))

    disp = ConfusionMatrixDisplay(cm, display_labels=["legitimate", "phishing"])
    disp.plot(values_format="d")
    plt.title(f"Confusion Matrix — {name}")
    plt.show()

    return result, model

results = []
fitted_models = {}

for name, model in models.items():
    result, fitted = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results.append(result)
    fitted_models[name] = fitted

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
display(results_df)

## 10. Error Analysis

False Negative is especially dangerous in phishing detection because a phishing URL is classified as legitimate.  
False Positive is also problematic because a legitimate URL is blocked, but it is usually less dangerous than missing a phishing attack.

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]

y_pred = best_model.predict(X_test)

test_errors = X_test.copy()
test_errors["true_label"] = y_test.values
test_errors["pred_label"] = y_pred
test_errors["url"] = df.loc[X_test.index, "url"].values
test_errors["status"] = df.loc[X_test.index, "status"].values

false_positives = test_errors[(test_errors["true_label"] == 0) & (test_errors["pred_label"] == 1)]
false_negatives = test_errors[(test_errors["true_label"] == 1) & (test_errors["pred_label"] == 0)]

print("Best model:", best_model_name)
print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

display(false_positives[["url", "status", "true_label", "pred_label"]].head(10))
display(false_negatives[["url", "status", "true_label", "pred_label"]].head(10))

## 11. Save Results

In [ ]:
Path("results").mkdir(exist_ok=True)
results_df.to_csv("results/model_metrics.csv", index=False)
print("Saved results/model_metrics.csv")

## 12. Initial Conclusion

The dataset is balanced and contains no missing values. The models can be trained successfully on the engineered URL and webpage features. Tree-based ensemble models are expected to perform strongly because they can capture non-linear relationships between URL characteristics and phishing labels.

The original GitHub project remains useful as the selected source, but our report must clearly state that the original processed CSV file was missing, so the exact original experiment could not be reproduced directly.